# Reproducing the ten-symbol trend sweep on streamed data

The trend lane is moving to a wider universe streamed straight from the Binance archive. Before any wide number is trusted, the new pipeline has to return the old numbers on the old window. If it does not, every result after it inherits the fault.

The old pipeline read one-minute CSVs from disk and has since been deleted. What survives is the numbers it recorded. So the test is this. Stream the same ten symbols over 2022 to 2024, run the same method unchanged, and compare against the recorded grid.

The old daily close was the close of the last one-minute bar of each day. A daily kline closes at the same instant with the same price, so the match should be exact.


## Pulling the panel

Stream daily klines and funding for the ten perps from 2022-01-01 up to 2025-01-01. Closes are dated by the bar's close time. Any day missing inside a symbol's span is refilled from the daily archive.


In [1]:
import numpy as np
import pandas as pd
import stream

SYMBOLS = ['ADAUSDT', 'AVAXUSDT', 'BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'ETHUSDT', 'LINKUSDT', 'LTCUSDT', 'SOLUSDT', 'XRPUSDT']
START, END = '2022-01-01', '2025-01-01'

daily, kline_reports = stream.daily_closes(SYMBOLS, START, END)
funding, funding_reports = stream.funding(SYMBOLS, START, END)

print(f'daily {daily.shape}  {daily.index.min().date()} .. {daily.index.max().date()}  missing cells {int(daily.isna().sum().sum())}')
print(f'funding {funding.shape}  one row per 8 hours')
print(pd.DataFrame([(r.symbol, str(r.kind).split('.')[-1], r.stats.files_read, r.stats.files_planned, r.stats.rows_parsed, r.stats.rows_rejected)
                    for r in kline_reports + funding_reports],
                   columns=['symbol', 'kind', 'files_read', 'files_planned', 'rows', 'rejected']).to_string(index=False))


LTCUSDT filled 2022-02-26 to 2022-02-28 from daily files, 3 rows
LTCUSDT filled 2022-04-01 to 2022-04-02 from daily files, 2 rows
SOLUSDT filled 2022-02-26 to 2022-02-28 from daily files, 3 rows
SOLUSDT filled 2022-04-01 to 2022-04-02 from daily files, 2 rows
XRPUSDT filled 2022-02-26 to 2022-02-28 from daily files, 3 rows
XRPUSDT filled 2022-04-01 to 2022-04-02 from daily files, 2 rows
daily (1096, 10)  2022-01-01 .. 2024-12-31  missing cells 0
funding (3363, 10)  one row per 8 hours
  symbol        kind  files_read  files_planned  rows  rejected
 ADAUSDT      Klines          36             36  1096         0
AVAXUSDT      Klines          36             36  1096         0
 BNBUSDT      Klines          36             36  1096         0
 BTCUSDT      Klines          36             36  1096         0
DOGEUSDT      Klines          36             36  1096         0
 ETHUSDT      Klines          36             36  1096         0
LINKUSDT      Klines          36             36  1096         

Every planned file was read and no row was rejected. The panel is 1096 days by ten symbols with no gaps. The funding panel is 3363 rows, the same shapes the old pipeline had. Most names print 3288 times, and SOL prints 3363 because it funded every two hours from 2022-11-09 to 11-18, the week FTX collapsed.

Three symbols needed a refill. The monthly daily-kline files for LTC, SOL and XRP leave out 2022-02-26 to 02-28 and 2022-04-01 to 04-02. The daily files for those same days exist. 02-27 is a Sunday, so without the refill each of those names would take its weekly close from the Friday.


## The archived method

The method is copied unchanged. Weekly close is the last daily close of each week ending Sunday. The signal is the sign of the trailing $L$-week return. Each name earns its sign times next week's return, minus next week's funding when long, plus it when short, minus its round-trip cost on each flip. The book is the equal-weight mean across names.

$$\text{net}_{i,t} = s_{i,t}\, r_{i,t+1} - s_{i,t}\, F_{i,t+1} - \mathbb{1}[s_{i,t} \ne s_{i,t-1}]\, c_i, \qquad s_{i,t} = \text{sign}\!\left(\frac{P_{i,t}}{P_{i,t-L}} - 1\right)$$

Reading the terms, one symbol $i$ in one week $t$.

- $s_{i,t}$ is the position, $+1$ long or $-1$ short, fixed by the sign of the return over the last $L$ weeks.
- $s_{i,t}\, r_{i,t+1}$ is that sign carried into next week's return, the raw trend payoff.
- $-s_{i,t}\, F_{i,t+1}$ is funding on the held position, paid out when long so it comes off, earned when short so it adds back.
- $\mathbb{1}[s_{i,t} \ne s_{i,t-1}]\, c_i$ is the round trip $c_i$, charged only on a week the sign flips from the week before.

$\text{net}_{i,t}$ is one name. The book is the flat mean of it across the ten.

The per-symbol costs are the round trips the old cost table produced, taker fee plus median half spread on each side. That table was deleted with the CSVs, so its numbers are carried here as constants.

In [2]:
costs = pd.Series({'ADAUSDT': 16.74, 'AVAXUSDT': 14.72, 'BNBUSDT': 14.41, 'BTCUSDT': 14.03, 'DOGEUSDT': 15.33,
                   'ETHUSDT': 14.05, 'LINKUSDT': 15.08, 'LTCUSDT': 15.39, 'SOLUSDT': 14.40, 'XRPUSDT': 15.79})

weekly = daily.resample('W-SUN').last()
fwd = weekly.pct_change().shift(-1)
fwd_funding = funding.resample('W-SUN').sum().reindex(weekly.index).shift(-1)

def book_sharpe(book):
    return book.mean() / book.std() * np.sqrt(52)

def trend_book(L, cost_mult=1.0, include_funding=True):
    net, gross, flips_all, fund_all = [], [], [], []
    for sym in weekly.columns:
        sig = np.sign(weekly[sym] / weekly[sym].shift(L) - 1.0)
        r = fwd[sym]
        idx = sig.dropna().index.intersection(r.dropna().index)
        sig, r = sig.loc[idx], r.loc[idx]
        f = fwd_funding[sym].reindex(idx).fillna(0.0)
        flips = (sig != sig.shift(1)).astype(float)
        g = sig * r
        fp = -sig * f if include_funding else pd.Series(0.0, index=idx)
        net.append((g + fp - flips * costs[sym] * cost_mult / 1e4).rename(sym))
        gross.append(g.rename(sym))
        flips_all.append(flips.rename(sym))
        fund_all.append(fp.rename(sym))
    return [pd.concat(x, axis=1) for x in (net, gross, flips_all, fund_all)]


## Lookback grid against the record

Run the five lookbacks with and without funding and put the streamed grid next to the recorded one. The difference table should be zero everywhere.


In [3]:
LOOKBACKS = [1, 2, 4, 8, 12]
rows = []
for L in LOOKBACKS:
    net, _, flips, fund = trend_book(L)
    book = net.mean(axis=1)
    rows.append({'L': L,
                 'sharpe_nofund': book_sharpe(trend_book(L, include_funding=False)[0].mean(axis=1)),
                 'sharpe_withfund': book_sharpe(book),
                 'mean_bps': book.mean() * 1e4,
                 'funding_drag_bps': fund.mean(axis=1).mean() * 1e4,
                 'flips_yr': flips.mean().mean() * 52})
streamed = pd.DataFrame(rows).set_index('L').round(2)

recorded = pd.DataFrame({'sharpe_nofund': [0.84, -0.09, 0.71, -0.14, 0.15],
                         'sharpe_withfund': [0.78, -0.14, 0.61, -0.24, 0.06],
                         'mean_bps': [73.82, -13.28, 58.70, -21.91, 6.37],
                         'funding_drag_bps': [-6.15, -4.01, -10.07, -8.61, -9.07],
                         'flips_yr': [24.97, 17.98, 12.17, 9.70, 8.54]},
                        index=pd.Index(LOOKBACKS, name='L'))

print('streamed'); print(streamed)
print('\nrecorded minus streamed'); print((recorded - streamed).round(2))


streamed
    sharpe_nofund  sharpe_withfund  mean_bps  funding_drag_bps  flips_yr
L                                                                       
1            0.84             0.78     73.82             -6.15     24.97
2           -0.09            -0.14    -13.28             -4.01     17.98
4            0.71             0.61     58.70            -10.07     12.17
8           -0.14            -0.24    -21.91             -8.61      9.70
12           0.15             0.06      6.37             -9.07      8.54

recorded minus streamed
    sharpe_nofund  sharpe_withfund  mean_bps  funding_drag_bps  flips_yr
L                                                                       
1             0.0              0.0       0.0               0.0       0.0
2             0.0              0.0       0.0               0.0       0.0
4             0.0              0.0       0.0               0.0       0.0
8             0.0              0.0       0.0               0.0       0.0
12            0.0

Every cell matches to the recorded precision. Sharpe with and without funding, mean return, funding drag and turnover all come back identical at every lookback.


## Cost levels, reference lines and per-symbol

Three more recorded outputs make the check stricter. The grid at zero, one and two times cost exercises the flip accounting. The long-only lines exercise the forward returns alone. The per-symbol Sharpe at $L=1$ exercises each name separately, which is where a single bad close would show.


In [4]:
cost_grid = pd.DataFrame({m: [book_sharpe(trend_book(L, cost_mult=m)[0].mean(axis=1)) for L in LOOKBACKS] for m in (0.0, 1.0, 2.0)},
                         index=pd.Index(LOOKBACKS, name='L')).round(2)
cost_recorded = pd.DataFrame({0.0: [0.86, -0.08, 0.65, -0.21, 0.09], 1.0: [0.78, -0.14, 0.61, -0.24, 0.06], 2.0: [0.71, -0.19, 0.57, -0.27, 0.04]},
                             index=pd.Index(LOOKBACKS, name='L'))
print('cost grid, recorded minus streamed'); print((cost_recorded - cost_grid).round(2))

lo = fwd.mean(axis=1).dropna()
vol = weekly.pct_change().rolling(8).std()
lo_vs = (fwd * (0.01 / vol).clip(upper=3.0).shift(1)).mean(axis=1).dropna()
print(f'\nlong-only {book_sharpe(lo):.2f} (recorded 0.54)   long-only vol-scaled {book_sharpe(lo_vs):.2f} (recorded 0.87)')

per_symbol = trend_book(1)[0].apply(book_sharpe).round(2)
per_recorded = pd.Series({'ADAUSDT': 0.26, 'AVAXUSDT': 1.17, 'BNBUSDT': 0.15, 'BTCUSDT': 0.01, 'DOGEUSDT': 1.12,
                          'ETHUSDT': 1.05, 'LINKUSDT': -0.45, 'LTCUSDT': 0.26, 'SOLUSDT': -0.03, 'XRPUSDT': 0.65})
print('\nper-symbol Sharpe at L=1')
print(pd.DataFrame({'recorded': per_recorded, 'streamed': per_symbol, 'diff': (per_recorded - per_symbol).round(2)}))


cost grid, recorded minus streamed
    0.0  1.0  2.0
L                
1   0.0  0.0  0.0
2   0.0  0.0  0.0
4   0.0  0.0  0.0
8   0.0  0.0  0.0
12  0.0  0.0  0.0

long-only 0.54 (recorded 0.54)   long-only vol-scaled 0.87 (recorded 0.87)

per-symbol Sharpe at L=1
          recorded  streamed  diff
ADAUSDT       0.26      0.26   0.0
AVAXUSDT      1.17      1.17   0.0
BNBUSDT       0.15      0.15   0.0
BTCUSDT       0.01      0.01   0.0
DOGEUSDT      1.12      1.12   0.0
ETHUSDT       1.05      1.05   0.0
LINKUSDT     -0.45     -0.45   0.0
LTCUSDT       0.26      0.26   0.0
SOLUSDT      -0.03     -0.03   0.0
XRPUSDT       0.65      0.65   0.0


All three match exactly. The cost grid is identical at every multiplier, long-only lands on 0.54 and 0.87, and every symbol's Sharpe agrees.

Without the refilled days the match breaks in exactly one place. SOL comes back at $-0.09$ against $-0.03$ and XRP at 0.62 against 0.65, which moves the book by about 0.01 to 0.02 at $L=1$, $L=4$ and $L=8$. The seven names with complete monthly files match either way. The gap is the archive, not the method.


## How the source stamps and bounds what it streams

Two behaviours of the source matter for everything after this and do not show up in a pandas panel keyed on close time. Pull one symbol for January 2022 with the window ending exactly on a month boundary and read the raw stamps and the bar identity each event carries.


In [5]:
q = stream.module()
um = int(q.BinanceMarket.UsdM)
b = q.SubscriptionBuilder(); b.add(q.ExchangeId.Binance, um, 'BTCUSDT'); sub = b.build()
cfg = q.BinanceHistoricalConfig([q.StreamSpec(q.EndpointKind.Klines, '1d')], q.Cadence.Monthly,
                                stream._ns('2022-01-01'), stream._ns('2022-02-01'), q.FetchPoolConfig())
bt = q.PythonBacktest(sub, cfg)
raw = []
bt.set_on_event(lambda e: raw.append((e.base.ts, e.payload.open_time, e.payload.close)))
bt.plan(); bt.run()
raw = pd.DataFrame(raw, columns=['event_ts', 'open_time', 'close'])
for col in ('event_ts', 'open_time'):
    raw[col] = pd.to_datetime(raw[col], unit='ns', utc=True)
print(raw.head(2).to_string(index=False))
print(f'\nfiles read {bt.reports()[0].stats.files_read}   bars {len(raw)}   last bar closes {raw.event_ts.max()}')


                        event_ts                 open_time    close
2022-01-01 23:59:59.999000+00:00 2022-01-01 00:00:00+00:00 47704.35
2022-01-02 23:59:59.999000+00:00 2022-01-02 00:00:00+00:00 47280.00

files read 2   bars 59   last bar closes 2022-02-28 23:59:59.999000+00:00


A kline event is stamped at the bar's close and carries its open time as the bar's identity. The 2022-01-01 bar arrives at 23:59:59.999 holding that day's close, so nothing driven by event time sees a close before it was printed. The loader keys each day on the event stamp.

The window selects files, not rows. A window ending on 2022-02-01 also read the February file, 59 bars instead of 31, because a file is kept when its first day falls inside the window at either end. The loader stops one nanosecond short of the end and clips rows itself.


## Verdict

The streamed pipeline reproduces the recorded ten-symbol sweep exactly, on every grid, reference line and symbol, once the days the monthly files drop are refilled from the daily files. The wide run can be built on it.
